In [0]:
dbutils.fs.mkdirs("/Volumes/sunil_catalog/assignment3/medallion/bronze")
dbutils.fs.mkdirs("/Volumes/sunil_catalog/assignment3/medallion/silver")
dbutils.fs.mkdirs("/Volumes/sunil_catalog/assignment3/medallion/gold")

True

In [0]:
display(dbutils.fs.ls("/Volumes/sunil_catalog/assignment3/medallion"))

path,name,size,modificationTime
dbfs:/Volumes/sunil_catalog/assignment3/medallion/Assignment_3_Sample_Sales_Data.csv,Assignment_3_Sample_Sales_Data.csv,415,1781100467000
dbfs:/Volumes/sunil_catalog/assignment3/medallion/bronze/,bronze/,0,1781108574676
dbfs:/Volumes/sunil_catalog/assignment3/medallion/gold/,gold/,0,1781108574676
dbfs:/Volumes/sunil_catalog/assignment3/medallion/silver/,silver/,0,1781108574676


In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/sunil_catalog/assignment3/medallion/Assignment_3_Sample_Sales_Data.csv")

display(df)

order_id,order_date,region,product,quantity,price
2001,2025-12-06,South,Tablet,6,180.0
2002,2025-12-06,North,Laptop,1,1300.0
2003,2025-12-07,East,Monitor,4,250.0
2004,2025-12-07,West,Mouse,30,25.0
2005,2025-12-08,South,Laptop,2,999.0
2006,2025-12-08,East,Dock,8,155.0
2007,2025-12-09,North,Chair,7,175.0
2008,2025-12-09,West,Laptop,1,1100.0
2009,2025-12-10,South,Headset,15,80.0
2010,2025-12-10,East,Tablet,3,210.0


In [0]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)



In [0]:
df.select("order_id", "product", "price").show()

+--------+-------+------+
|order_id|product| price|
+--------+-------+------+
|    2001| Tablet| 180.0|
|    2002| Laptop|1300.0|
|    2003|Monitor| 250.0|
|    2004|  Mouse|  25.0|
|    2005| Laptop| 999.0|
|    2006|   Dock| 155.0|
|    2007|  Chair| 175.0|
|    2008| Laptop|1100.0|
|    2009|Headset|  80.0|
|    2010| Tablet| 210.0|
+--------+-------+------+



In [0]:
from pyspark.sql.functions import col

df2 = df.withColumn(
    "sales_amount",
    col("quantity") * col("price")
)

display(df2)

order_id,order_date,region,product,quantity,price,sales_amount
2001,2025-12-06,South,Tablet,6,180.0,1080.0
2002,2025-12-06,North,Laptop,1,1300.0,1300.0
2003,2025-12-07,East,Monitor,4,250.0,1000.0
2004,2025-12-07,West,Mouse,30,25.0,750.0
2005,2025-12-08,South,Laptop,2,999.0,1998.0
2006,2025-12-08,East,Dock,8,155.0,1240.0
2007,2025-12-09,North,Chair,7,175.0,1225.0
2008,2025-12-09,West,Laptop,1,1100.0,1100.0
2009,2025-12-10,South,Headset,15,80.0,1200.0
2010,2025-12-10,East,Tablet,3,210.0,630.0


In [0]:
df3 = df.withColumnRenamed(
    "price",
    "unit_price"
)

In [0]:
df2.filter(df2.sales_amount > 1000).show()

+--------+----------+------+-------+--------+------+------------+
|order_id|order_date|region|product|quantity| price|sales_amount|
+--------+----------+------+-------+--------+------+------------+
|    2001|2025-12-06| South| Tablet|       6| 180.0|      1080.0|
|    2002|2025-12-06| North| Laptop|       1|1300.0|      1300.0|
|    2005|2025-12-08| South| Laptop|       2| 999.0|      1998.0|
|    2006|2025-12-08|  East|   Dock|       8| 155.0|      1240.0|
|    2007|2025-12-09| North|  Chair|       7| 175.0|      1225.0|
|    2008|2025-12-09|  West| Laptop|       1|1100.0|      1100.0|
|    2009|2025-12-10| South|Headset|      15|  80.0|      1200.0|
+--------+----------+------+-------+--------+------+------------+



In [0]:
df2.filter(df2.product == "Laptop").show()

+--------+----------+------+-------+--------+------+------------+
|order_id|order_date|region|product|quantity| price|sales_amount|
+--------+----------+------+-------+--------+------+------------+
|    2002|2025-12-06| North| Laptop|       1|1300.0|      1300.0|
|    2005|2025-12-08| South| Laptop|       2| 999.0|      1998.0|
|    2008|2025-12-09|  West| Laptop|       1|1100.0|      1100.0|
+--------+----------+------+-------+--------+------+------------+



find orders with sales greater then 1000GBP

In [0]:
from pyspark.sql.functions import sum

sales_by_region = df2.groupBy("region") \
    .agg(sum("sales_amount").alias("total_sales"))

display(sales_by_region)

region,total_sales
North,2525.0
East,2870.0
South,4278.0
West,1850.0


Calculate average price by product:

In [0]:
from pyspark.sql.functions import avg

avg_price = df.groupBy("product") \
    .agg(avg("price").alias("avg_price"))

display(avg_price)

product,avg_price
Mouse,25.0
Chair,175.0
Tablet,195.0
Monitor,250.0
Headset,80.0
Laptop,1133.0
Dock,155.0


In [0]:
print(type(df))

<class 'pyspark.sql.connect.dataframe.DataFrame'>


In [0]:
df.write.mode("overwrite").parquet(
    "/Volumes/sunil_catalog/assignment3/medallion/bronze/sales_data"
)

In [0]:
df2.write.mode("overwrite").parquet(
    "/Volumes/sunil_catalog/assignment3/medallion/silver/sales_data"
)

In [0]:
sales_by_region.write.mode("overwrite").parquet(
    "/Volumes/sunil_catalog/assignment3/medallion/gold/sales_summary"
)

In [0]:
bronze_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/sunil_catalog/assignment3/medallion/Assignment_3_Sample_Sales_Data.csv")

display(bronze_df)

order_id,order_date,region,product,quantity,price
2001,2025-12-06,South,Tablet,6,180.0
2002,2025-12-06,North,Laptop,1,1300.0
2003,2025-12-07,East,Monitor,4,250.0
2004,2025-12-07,West,Mouse,30,25.0
2005,2025-12-08,South,Laptop,2,999.0
2006,2025-12-08,East,Dock,8,155.0
2007,2025-12-09,North,Chair,7,175.0
2008,2025-12-09,West,Laptop,1,1100.0
2009,2025-12-10,South,Headset,15,80.0
2010,2025-12-10,East,Tablet,3,210.0


In [0]:
bronze_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/sunil_catalog/assignment3/medallion/bronze/sales_data")

In [0]:
display(
    spark.read.parquet(
        "/Volumes/sunil_catalog/assignment3/medallion/bronze/sales_data"
    )
)

order_id,order_date,region,product,quantity,price
2001,2025-12-06,South,Tablet,6,180.0
2002,2025-12-06,North,Laptop,1,1300.0
2003,2025-12-07,East,Monitor,4,250.0
2004,2025-12-07,West,Mouse,30,25.0
2005,2025-12-08,South,Laptop,2,999.0
2006,2025-12-08,East,Dock,8,155.0
2007,2025-12-09,North,Chair,7,175.0
2008,2025-12-09,West,Laptop,1,1100.0
2009,2025-12-10,South,Headset,15,80.0
2010,2025-12-10,East,Tablet,3,210.0


In [0]:
silver_df = bronze_df.toDF(
    "order_id",
    "order_date",
    "region",
    "product",
    "quantity",
    "price"
)

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.withColumn(
    "total_price",
    col("quantity") * col("price")
)
display(silver_df)

order_id,order_date,region,product,quantity,price,total_price
2001,2025-12-06,South,Tablet,6,180.0,1080.0
2002,2025-12-06,North,Laptop,1,1300.0,1300.0
2003,2025-12-07,East,Monitor,4,250.0,1000.0
2004,2025-12-07,West,Mouse,30,25.0,750.0
2005,2025-12-08,South,Laptop,2,999.0,1998.0
2006,2025-12-08,East,Dock,8,155.0,1240.0
2007,2025-12-09,North,Chair,7,175.0,1225.0
2008,2025-12-09,West,Laptop,1,1100.0,1100.0
2009,2025-12-10,South,Headset,15,80.0,1200.0
2010,2025-12-10,East,Tablet,3,210.0,630.0


In [0]:
silver_df = silver_df.filter(
    col("total_price") > 1000
)

display(silver_df)

order_id,order_date,region,product,quantity,price,total_price
2001,2025-12-06,South,Tablet,6,180.0,1080.0
2002,2025-12-06,North,Laptop,1,1300.0,1300.0
2005,2025-12-08,South,Laptop,2,999.0,1998.0
2006,2025-12-08,East,Dock,8,155.0,1240.0
2007,2025-12-09,North,Chair,7,175.0,1225.0
2008,2025-12-09,West,Laptop,1,1100.0,1100.0
2009,2025-12-10,South,Headset,15,80.0,1200.0


In [0]:
silver_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/sunil_catalog/assignment3/medallion/silver/sales_data")

In [0]:
display(
    spark.read.parquet(
        "/Volumes/sunil_catalog/assignment3/medallion/silver/sales_data"
    )
)

order_id,order_date,region,product,quantity,price,total_price
2001,2025-12-06,South,Tablet,6,180.0,1080.0
2002,2025-12-06,North,Laptop,1,1300.0,1300.0
2005,2025-12-08,South,Laptop,2,999.0,1998.0
2006,2025-12-08,East,Dock,8,155.0,1240.0
2007,2025-12-09,North,Chair,7,175.0,1225.0
2008,2025-12-09,West,Laptop,1,1100.0,1100.0
2009,2025-12-10,South,Headset,15,80.0,1200.0


In [0]:
from pyspark.sql.functions import sum, count

gold_df = silver_df.groupBy("region") \
    .agg(
        sum("total_price").alias("total_sales"),
        count("order_id").alias("order_count")
    )

display(gold_df)

region,total_sales,order_count
North,2525.0,2
East,1240.0,1
South,4278.0,3
West,1100.0,1


In [0]:
gold_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/sunil_catalog/assignment3/medallion/gold/sales_summary")

In [0]:
display(
    spark.read.parquet(
        "/Volumes/sunil_catalog/assignment3/medallion/gold/sales_summary"
    )
)

region,total_sales,order_count
North,2525.0,2
East,1240.0,1
South,4278.0,3
West,1100.0,1
